# Lead AI Product Engineer: Claude & FastAPI Practice

This notebook demonstrates how to orchestrate the Claude API with structured JSON outputs and simulate a FastAPI backend.

In [ ]:
!pip install -qU anthropic pydantic fastapi uvicorn nest-asyncio

## 1. Claude API & Pydantic Validation
Ensuring the LLM returns strict JSON as required for business software.

In [ ]:
import os
from pydantic import BaseModel, Field
from typing import List
import anthropic

os.environ["ANTHROPIC_API_KEY"] = "your_key"

class SaaSProduct(BaseModel):
    name: str = Field(description="Name of the SaaS product")
    tagline: str = Field(description="Catchy tagline")
    features: List[str] = Field(description="List of 3 core features")
    pricing_tier: str = Field(description="Recommended starting price")

client = anthropic.Anthropic()

def generate_saas_idea(niche: str):
    response = client.messages.create(
        model="claude-3-haiku-20240307",
        max_tokens=1000,
        system="Return ONLY a JSON object matching the requested schema. No conversational text.",
        messages=[{"role": "user", "content": f"Generate a SaaS idea for the {niche} industry."}]
    )
    return response.content[0].text

# Mock run (since API key is missing)
print("System Prompt set to enforce strict JSON.")
print("Pydantic will validate the content: SaaSProduct.model_validate_json(raw_text)")

## 2. Mocking a FastAPI SaaS Backend
Simulating multi-tenancy and data isolation.

In [ ]:
from fastapi import FastAPI, HTTPException, Header

app = FastAPI()

# Mock Database
database = {
    "tenant_1": {"data": "Sensitive HR data for Apple"},
    "tenant_2": {"data": "Sensitive Billing data for Google"}
}

@app.get("/data")
async def get_tenant_data(x_tenant_id: str = Header(None)):
    if not x_tenant_id or x_tenant_id not in database:
        raise HTTPException(status_code=403, detail="Access Denied: Invalid Tenant ID")
    
    return {"data": database[x_tenant_id]["data"]}

print("FastAPI structure ready. In a real interview, explain how you'd use Middleware to handle this globally.")

## 3. Stripe Logic (Simulation)
Handling usage-based billing.

In [ ]:
def calculate_usage_bill(tokens_used: int, price_per_1k: float = 0.02):
    bill = (tokens_used / 1000) * price_per_1k
    return round(bill, 2)

print(f"Bill for 50,000 tokens: ${calculate_usage_bill(50000)}")

## Practice Exercises
1. **Pydantic:** Add an `email` field to `SaaSProduct` and ensure it's a valid email format.
2. **API Design:** Add a POST endpoint to the FastAPI mock that allows adding new data to a specific tenant.
3. **Claude:** Modify the prompt to include "Negative Constraints" (e.g., "Do not mention AI in the name").